# NAVI Fine-tuning — Qwen2.5-3B-Instruct

**Antes de empezar:** Activa la GPU en Colab:  
`Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`

Luego sube estos 4 archivos con el botón de carpeta (panel izquierdo → icono upload):
- `navi_finetune.py`
- `sft_rich.jsonl`
- `sft_sephirot.jsonl`
- `dpo_pairs.jsonl`

In [ ]:
# CELDA 1 — Verificar GPU
!nvidia-smi

In [ ]:
# CELDA 2 — Instalar dependencias (~3 min)
!pip install -q \
    torch \
    transformers>=4.45.0 \
    peft>=0.12.0 \
    trl>=0.12.0 \
    datasets>=2.20.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.30.0

print('✓ Dependencias instaladas')

In [ ]:
# CELDA 3 — Verificar archivos subidos
import os

archivos = [
    'navi_finetune.py',
    'sft_rich.jsonl',
    'sft_sephirot.jsonl',
    'dpo_pairs.jsonl',
]

for f in archivos:
    existe = os.path.exists(f)
    lineas = sum(1 for _ in open(f)) if existe and f.endswith('.jsonl') else '-'
    print(f"{'✓' if existe else '✗ FALTA'} {f}  {'(' + str(lineas) + ' muestras)' if lineas != '-' else ''}")

In [ ]:
# CELDA 4 — FASE 1: SFT LoRA (~20-40 min en T4)
# sft_sephirot va primero como base sintética de calidad
!python navi_finetune.py \
    --train \
    --data sft_rich.jsonl \
    --sephirot sft_sephirot.jsonl \
    --out ./lora_out

In [ ]:
# CELDA 5 — Verificar que el SFT terminó bien
!ls -lh ./lora_out/

In [ ]:
# CELDA 6 — FASE 2: DPO (~5-10 min en T4)
!python navi_finetune.py \
    --dpo \
    --sft-lora ./lora_out \
    --dpo-data dpo_pairs.jsonl \
    --out ./lora_dpo_out

In [ ]:
# CELDA 7 — Verificar DPO
!ls -lh ./lora_dpo_out/

In [ ]:
# CELDA 8 — Descargar los pesos LoRA (solo unos MB)
# Estos los llevas a tu máquina para el merge + GGUF local
import shutil

shutil.make_archive('lora_dpo_out', 'zip', '.', 'lora_dpo_out')
print('✓ lora_dpo_out.zip listo')

# Descarga automática en el navegador
from google.colab import files
files.download('lora_dpo_out.zip')

## Siguiente paso — en tu máquina local

Una vez descargado `lora_dpo_out.zip`, descomprímelo y ejecuta el merge en CPU:  
(no necesita GPU, solo ~12 GB de RAM libre)

```bash
# Merge LoRA → modelo completo
python navi_finetune.py --merge --lora ./lora_dpo_out --out ./merged

# Convertir a GGUF Q4_K_M para llama.cpp
python navi_finetune.py --to-gguf ./merged --out ./gguf_out
```

El GGUF final (`navi-3b-q4_k_m.gguf`) es el que cargas en llama-server como ahora.